# harness-admission: GSM-Hard, clean start (no handicap)
Same pipeline as the GSM8K run, on GSM-Hard (TRPrompt's benchmark), starting from the normal prompt. Results go to `ha_home_gsmhard`, separate from the GSM8K run.


Run the cells **in order, one at a time**. Every step is resumable: after a disconnect or restart, run **cell 1** again, then the step you were on. Finished work is skipped.

Rules:
- **Never press stop** while a model is loading or generating.
- **Never load a model inside the notebook** (`LLM(...)`). All model runs go through `run(...)`, which frees the GPU first.
- Keep the same GPU type for the whole experiment (cell 1 checks this).

In [1]:
# ===== CELL 1: setup (run this first, and again after every restart) =====
import os, re, subprocess, sys, time, json, glob

REPO_URL = "https://github.com/Soumya080/harness-admission.git"
REPO = os.environ.get("HA_REPO_DIR", "/content/ha")
os.environ["HA_DATASET"] = "gsmhard"   # GSM-Hard (TRPrompt's benchmark)
os.environ["HA_REGIME"] = "clean"      # realistic start: no handicap, no planted edit

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout.strip()

# 1. packages (only installs if vLLM is missing)
try:
    import vllm  # noqa
except ImportError:
    print("installing packages (5-10 min)...")
    sh(f"{sys.executable} -m pip install -q vllm datasets pandas pyarrow scipy matplotlib pytest")
    print("installed. If the next cells fail on import, do Runtime > Restart session and run this cell again.")

# 2. code
if os.path.isdir(os.path.join(REPO, ".git")):
    sh(f"cd {REPO} && git pull -q", check=False)
elif not os.path.isdir(REPO):
    sh(f"git clone -q {REPO_URL} {REPO}")
os.chdir(REPO)
assert os.path.isdir("tests") and os.path.isdir("scripts"), "wrong repo: tests/ or scripts/ missing"

# 3. where results live: Google Drive if it mounts, else local disk
if "HA_HOME" not in os.environ or not os.path.isdir(os.environ["HA_HOME"]):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", str(e)[:80])
    home = "/content/drive/MyDrive/ha_home_gsmhard" if os.path.isdir("/content/drive/MyDrive") else "/content/ha_home_gsmhard"
    os.makedirs(home, exist_ok=True)
    os.environ["HA_HOME"] = home
HOME = os.environ["HA_HOME"]
ON_DRIVE = os.path.ismount("/content/drive") and HOME.startswith("/content/drive")
os.environ.setdefault("HA_DTYPE", "half")
os.environ.setdefault("HA_GPU_UTIL", "0.85")
os.makedirs(f"{HOME}/logs", exist_ok=True); os.makedirs(f"{HOME}/results", exist_ok=True)

# 4. GPU helpers
def gpu_name():
    try: return sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines()[0].strip()
    except Exception: return "NO GPU"

def gpu_used():
    try: return int(sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits").split()[0])
    except Exception: return 0

def free_gpu(limit=1500):
    """Kill every leftover process holding GPU memory. Raises if the GPU cannot be freed."""
    if gpu_used() < limit: return
    pids = sh("nvidia-smi --query-compute-apps=pid --format=csv,noheader", check=False).split()
    for p in pids:
        if p.isdigit() and int(p) != os.getpid():
            sh(f"kill -9 {p}", check=False)
    for pat in ("VLLM", "EngineCore", "scripts.run_grid", "proposer.propose"):
        sh(f"pkill -9 -f '{pat}'", check=False)
    for _ in range(10):
        time.sleep(2)
        if gpu_used() < limit: print("GPU freed."); return
    raise RuntimeError(f"GPU still has {gpu_used()} MiB in use. Do Runtime > Restart session, "
                       "then run cell 1 again. (A model was loaded inside the notebook.)")

NOISE = re.compile(r"EngineCore pid|MainProcess pid|torch/_|Downloading|Reconstruct|Fetching|safetensors|"
                   r"unauthenticated|it/s|Generating .* split|README.md|\.parquet:|config\.json|Download complete")

def run(cmd, gpu=True, save=None):
    """Run a repo command in its own process. Shows only the useful lines; full log goes to HA_HOME/logs."""
    if gpu: free_gpu()
    log = f"{HOME}/logs/{time.strftime('%m%d_%H%M%S')}_{re.sub(r'[^a-z_]+', '_', cmd.lower())[:40]}.log"
    print(f"$ {cmd}")
    t0, shown = time.time(), []
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         cwd=REPO, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    with open(log, "w") as f:
        for line in p.stdout:
            f.write(line)
            if line.strip() and not NOISE.search(line):
                print("  " + line.rstrip()[:200]); shown.append(line)
    p.wait()
    if save:
        open(f"{HOME}/results/{save}", "w").writelines(shown)
    if p.returncode != 0:
        errs = [l.strip() for l in open(log) if re.search(r"Error|STOP|FAILED", l)]
        print("\nFAILED. Root cause lines:"); [print("  ", e[-300:]) for e in errs[-4:]]
        raise RuntimeError(f"step failed (full log: {log})")
    print(f"ok ({time.time() - t0:.0f}s)\n")

def harness_tags():
    out = {}
    for f in glob.glob(f"{HOME}/cache/harnesses/*.json"):
        t = json.load(open(f)).get("tag", "?"); out[t] = out.get(t, 0) + 1
    return out

# 5. one GPU type for the whole experiment
name = gpu_name(); gfile = f"{HOME}/gpu_name.txt"
if os.path.exists(gfile):
    old = open(gfile).read().strip()
    if old != name:
        print(f"!!! WARNING: earlier results were made on '{old}', this session has '{name}'.\n"
              f"!!! Do not continue the grid on a different GPU. Get '{old}' again, or delete {HOME}/cache/outcomes and rerun.")
elif name != "NO GPU":
    open(gfile, "w").write(name)

n_out = len(glob.glob(f"{HOME}/cache/outcomes/*.parquet"))
print(f"repo: {REPO}\nresults folder: {HOME}  ({'Google Drive: survives disconnects' if ON_DRIVE else 'LOCAL DISK: run the backup cell often'})")
print(f"GPU: {name}, {gpu_used()} MiB in use\nharnesses so far: {harness_tags()} | finished result files: {n_out}")

installing packages (5-10 min)...
installed. If the next cells fail on import, do Runtime > Restart session and run this cell again.
Drive not mounted: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See exa
repo: /content/ha
results folder: /content/ha_home_gsmhard  (LOCAL DISK: run the backup cell often)
GPU: Tesla T4, 0 MiB in use
harnesses so far: {} | finished result files: 0


In [2]:
# ===== CELL 2: CPU checks (about 5 min). Must end with "ALL DONE". =====
run("python -m pytest -q tests", gpu=False)
run("HA_FAKE=1 HA_HOME=$(mktemp -d) bash scripts/run_all.sh 2>&1 | tail -3", gpu=False)

$ python -m pytest -q tests
  ........................                                                 [100%]
  24 passed in 3.69s
ok (7s)

$ HA_FAKE=1 HA_HOME=$(mktemp -d) bash scripts/run_all.sh 2>&1 | tail -3
  }
  ALL DONE. Log: /tmp/tmp.EpUuadcqRj/logs/run_20261127_0910.log   Results: /tmp/tmp.EpUuadcqRj/results
ok (303s)



In [3]:
# ===== CELL 3: base + STOP CHECK (GPU, about 10 min) =====
if not os.path.exists(f"{HOME}/configs/pools_gsmhard.json"):
    run("python -m data.pools", gpu=False)
run("python -m proposer.manual", gpu=False)         # base (clean) + 5 rewordings; no planted edit
run("python -m scripts.run_grid --only base")
run("python -m scripts.check_stop", gpu=False)      # raises if the check fails

$ python -m data.pools
  gsmhard {'proposal': 100, 'dev': 200, 'audit': 400} -> /content/ha_home_gsmhard/configs/pools_gsmhard.json
ok (4s)

$ python -m proposer.manual
  base     467de61d65
  noop     b9c8c8f15f
  noop     64c2615148
  noop     61e957af05
  noop     9e82aae582
  noop     cdebc377c6
ok (0s)

$ python -m scripts.run_grid --only base
  1 harnesses
  Processed prompts:   4%|▎         | 7/200 [00:20<03:16,  1.02s/it, est. speed input: 28.07 toks/s, output: 45.50 toks/s]
  base     467de61d65 dev   t=0 acc=0.485 243s
  Processed prompts:   2%|▏         | 20/1000 [00:30<17:32,  1.07s/it, est. speed input: 53.05 toks/s, output: 105.48 toks/s]
  base     467de61d65 dev   t=0.7 acc=0.473 205s
  Processed prompts:   2%|▏         | 8/400 [00:25<07:17,  1.12s/it, est. speed input: 25.22 toks/s, output: 38.92 toks/s]
  base     467de61d65 audit t=0 acc=0.520 92s
  Processed prompts:   6%|▌         | 6/100 [00:12<01:50,  1.17s/it, est. speed input: 40.80 toks/s, output: 63.35 toks/s

**Continue only if cell 3 printed `CHECK PASSED`.**

In [4]:
# ===== CELL 4: proposer, 30 LLM edits (GPU, 7B model, about 10-20 min) =====
n = harness_tags().get("llm", 0)
print("llm edits so far:", n)
if n < 30:
    run(f"python -m proposer.propose --n {30 - n}")
print(harness_tags())      # expect base 1, noop 5, llm 30

llm edits so far: 0
$ python -m proposer.propose --n 30
  51 proposal-pool failures
  Processed prompts: 100%|██████████| 8/8 [00:57<00:00,  7.19s/it, est. speed input: 513.40 toks/s, output: 9.11 toks/s]
  made 30 edits; parse success 30/32
ok (405s)

{'noop': 5, 'llm': 30, 'base': 1}


In [5]:
# ===== CELL 5: full grid (GPU, the long step). Rerun this cell if it stops. =====
run("python -m scripts.run_grid")
run("python -m scripts.diagnose", gpu=False, save="diagnose.txt")

$ python -m scripts.run_grid
  36 harnesses
  Processed prompts:   4%|▍         | 9/200 [00:29<03:15,  1.02s/it, est. speed input: 26.70 toks/s, output: 55.88 toks/s]
  noop     9e82aae582 dev   t=0 acc=0.520 153s
  Processed prompts:   3%|▎         | 30/1000 [00:50<16:46,  1.04s/it, est. speed input: 50.95 toks/s, output: 131.51 toks/s]
  noop     9e82aae582 dev   t=0.7 acc=0.495 253s
  Processed prompts:   2%|▏         | 9/400 [00:34<07:24,  1.14s/it, est. speed input: 21.14 toks/s, output: 47.00 toks/s]
  noop     9e82aae582 audit t=0 acc=0.540 111s
  Processed prompts:   4%|▍         | 8/200 [00:30<03:49,  1.19s/it, est. speed input: 24.81 toks/s, output: 48.47 toks/s]
  llm      5a5fcb3112 dev   t=0 acc=0.475 63s
  Processed prompts:   3%|▎         | 30/1000 [00:50<17:17,  1.07s/it, est. speed input: 56.12 toks/s, output: 141.13 toks/s]
  llm      5a5fcb3112 dev   t=0.7 acc=0.445 266s
  Processed prompts:   2%|▎         | 10/400 [00:38<06:47,  1.04s/it, est. speed input: 24.59 tok

In [6]:
# ===== CELL 6: stage-0 simulations + report + ablations (CPU, about 20-40 min) =====
run("python -m sims.stage0", gpu=False, save="stage0.txt")
run("python -m sims.plot_stage0", gpu=False)
run("python -m scripts.make_report --runs 1000 --ablations", gpu=False, save="report.txt")
run("python -m scripts.claimed_vs_real", gpu=False, save="claimed_vs_real.txt")
run("python -m scripts.selection_resolution", gpu=False, save="selection_resolution.txt")
print(sorted(os.listdir(f"{HOME}/results")))

$ python -m sims.stage0
  false-commit rate (PACE, peeking t-test): (0.0355, 0.266)
  best fake gain by dev n: {20: np.float64(14.5), 40: np.float64(10.2), 100: np.float64(6.5), 200: np.float64(4.6)}
  g at lam=0.5: {0.6: np.float64(-0.034), 0.7: np.float64(0.0759), 0.8: np.float64(0.1857)}
ok (216s)

$ python -m sims.plot_stage0
  saved /content/ha_home_gsmhard/results/fig_lambda.png
ok (1s)

$ python -m scripts.make_report --runs 1000 --ablations
  base audit acc 0.520 | best edit 0.557 | edits 35
      regime  n_dev    gate  commits  commits_sd  helpful_pct  null_pct  harmful_pct  final_gain  final_gain_sd   cost
  controlled     20  greedy      1.9         1.0          4.6      92.4          3.1         0.0            1.6  700.0
  controlled     20 mcnemar      0.1         0.3          1.2      98.8          0.0         0.0            0.4  700.0
  controlled     20    pace      0.0         0.0          0.0       0.0          0.0         0.0            0.0  700.0
  controlled     40

In [7]:
# ===== CELL 7: backup. Run after cells 3, 4, 5 and 6 if results are on LOCAL DISK. =====
zip_path = "/content/ha_home_gsmhard_backup.zip"
sh(f"cd {os.path.dirname(HOME)} && zip -qr {zip_path} {os.path.basename(HOME)}")
print("backup:", zip_path, f"{os.path.getsize(zip_path)/1e6:.1f} MB")
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("download it from the file sidebar:", e)

backup: /content/ha_home_gsmhard_backup.zip 20.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Restoring a backup in a new session
Upload `ha_home_gsmhard_backup.zip` with the file sidebar, then run:
```python
import subprocess; subprocess.run("mkdir -p /content && cd /content && unzip -qo ha_home_gsmhard_backup.zip", shell=True)
```
Then run cell 1 and continue from the step you were on.